v61 
- **Verified all metric calculation with Excel** 
- Updated core.analyzer
- Replaced the `Result` pattern with exceptions and flattened the logic
- Refactored the `AlphaEngine` into a streamlined Orchestrator pattern
-  **Strict Date Logic:** No more "Time Travel" bugs.
-  **Dataclass Contracts:** No more "Magic String" typos or blind dictionaries.
-  **Exception Flow:** The `run` method is now a clean, readable story instead of a maze of `if/else` checks.
-  **Performance Workers:** Math is separated from orchestration.
- Ret_1d, explicitly turns division-by-zero results (`inf`) into `NaN`, and replace [np.inf, -np.inf] with np.nan



v60  
- Converted code from notebook to modular system.
- Fixed divide by zero warning from calculate_gain
- Added subtitle to subplots
- Added Volatility Regime plot


v59  
- Removed "nest" of if-statements in **AlphaEngine.run**
- Use **Result Pattern** to handle errors
- Change verify_analyzer_short and verify_analyzer_long gain calculation from simple return to logarithmic return
- Change calculate_gain from simple return to logarithmic return
- Remove bfill from calculate_gain to prevent backfill with future data
- Verify macro_df calculation


In [1]:
# 1. Enable Autoreload
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

def add_project_root_to_path():
    """Find notebooks_RLVR and add to sys.path."""
    current = Path.cwd()

    # Search upward for notebooks_RLVR folder
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            print(f"✓ Added to path: {path}")
            return path
        # Also check if notebooks_RLVR exists as child (for running from stocks/)
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            print(f"✓ Added to path: {candidate}")
            return candidate

    raise RuntimeError("Could not find notebooks_RLVR directory")


# Run once at notebook start
add_project_root_to_path()


# 2. Force reload cached modules (run this to refresh code changes)
# import importlib

modules_to_reload = [
    "core.analyzer",
    "core.auditor",
    "core.contracts",    
    "core.engine",
    "core.features",
    "core.paths",
    "core.performance",
    "core.quant",
    "core.result",
    "core.settings",
    "core.utils",
    "strategy.registry",
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]


# 3. Standard imports
import pandas as pd
# import os
import numpy as np

from IPython.display import display
# from dataclasses import fields, asdict, is_dataclass
# from typing import List, Union, Tuple 


# 4. Fresh imports (these will re-import from disk due to cache clearing above)

from core.analyzer import create_walk_forward_analyzer
from core.auditor import SystemAuditor as SA
from core.contracts import FilterPack
from core.engine import AlphaEngine
from core.features import generate_features
from core.paths import OUTPUT_DIR
# from core.performance import PerformanceCalculator
# from core.quant import QuantUtils
# from core.result import TaskResult
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
# from strategy.registry import METRIC_REGISTRY


# 5. Pandas display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.precision", 4)


# # 6. Instantiate engine (customize DataFrames as needed)
# master_engine = AlphaEngine(
#     df_ohlcv=df_ohlcv,
#     features_df=features_df,
#     macro_df=macro_df,
#     df_close_wide=df_close_wide,
#     df_atrp_wide=df_atrp_wide,
#     df_trp_wide=df_trp_wide,
# )

✓ Added to path: c:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR
NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output



In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path


def load_env_from_root(env_var_name):
    """
    Load specified environment variable from .env file in root directory

    Parameters:
    -----------
    env_var_name : str
        Name of the environment variable to retrieve (e.g., 'DATA_PATH_OHLCV', 'API_KEY')

    Returns:
    --------
    str : Value of the requested environment variable

    Example:
    --------
    DATA_PATH_OHLCV = load_env_from_root('DATA_PATH_OHLCV')
    API_KEY = load_env_from_root('API_KEY')
    """
    # Start from current file or current working directory
    try:
        start_path = Path(__file__).resolve().parent
    except NameError:
        # If __file__ not defined (e.g., Jupyter), use current working directory
        start_path = Path.cwd()

    # Search upwards for the .env directory
    current = start_path
    for _ in range(10):  # Limit search depth
        env_path = current / ".env" / "my_api_key.env"
        if env_path.exists():
            load_dotenv(env_path, override=True)
            value = os.getenv(env_var_name)
            if value is None:
                raise KeyError(f"Variable '{env_var_name}' not found in {env_path}")
            return value

        # Go up one level
        parent = current.parent
        if parent == current:  # Reached root
            break
        current = parent

    raise FileNotFoundError(
        f"Could not find .env/my_api_key.env when searching for '{env_var_name}'"
    )


#

In [3]:
DATA_PATH_OHLCV = load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = load_env_from_root("DATA_PATH_INDICES")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")

DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


In [4]:
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
print(f"df_indices:|n{df_indices}")

df_indices:|n                   Adj Open  Adj High  Adj Low  Adj Close  Volume
Ticker Date                                                      
^AXJO  1992-11-22   1455.00   1455.00  1455.00    1455.00       0
       1992-11-23   1458.40   1458.40  1458.40    1458.40       0
       1992-11-24   1467.90   1467.90  1467.90    1467.90       0
       1992-11-25   1459.00   1459.00  1459.00    1459.00       0
       1992-11-26   1458.90   1458.90  1458.90    1458.90       0
...                     ...       ...      ...        ...     ...
^VIX3M 2026-03-06     26.08     27.75    24.99      27.56       0
       2026-03-09     28.04     29.09    24.86      25.34       0
       2026-03-10     24.94     25.84    23.54      25.51       0
       2026-03-11     25.29     25.98    24.93      24.97       0
       2026-03-12     26.27     26.99    25.80      26.95       0

[144407 rows x 5 columns]


In [5]:
_indices = df_indices.index.get_level_values(0).unique().tolist()
display(_indices)
df_indices.info()

['^AXJO',
 '^BSESN',
 '^DJI',
 '^FCHI',
 '^FTSE',
 '^GDAXI',
 '^GSPC',
 '^HSI',
 '^IXIC',
 '^N225',
 '^NYA',
 '^STOXX50E',
 '^VIX',
 '^VIX3M']

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 144407 entries, ('^AXJO', Timestamp('1992-11-22 00:00:00')) to ('^VIX3M', Timestamp('2026-03-12 00:00:00'))
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Adj Open   144407 non-null  float64
 1   Adj High   144407 non-null  float64
 2   Adj Low    144407 non-null  float64
 3   Adj Close  144407 non-null  float64
 4   Volume     144407 non-null  int64  
dtypes: float64(4), int64(1)
memory usage: 6.6+ MB


In [6]:
print(f"Takes about 1.5 minutes")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")

Takes about 1.5 minutes


In [7]:
print(f"df_ohlcv.head():\n {df_ohlcv.head()}\n")
df_ohlcv.info()

df_ohlcv.head():
                    Adj Open  Adj High  Adj Low  Adj Close    Volume
Ticker Date                                                        
A      1999-11-18   27.1966   29.8864  23.9091    26.3000  74849948
       1999-11-19   25.6649   25.7023  23.7970    24.1333  18230876
       1999-11-22   24.6936   26.3000  23.9465    26.3000   7871810
       1999-11-23   25.4034   26.0759  23.9091    23.9091   7151081
       1999-11-24   23.9838   25.0672  23.9091    24.5442   5795949

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 9503184 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-03-12 00:00:00'))
Data columns (total 5 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Adj Open   float64
 1   Adj High   float64
 2   Adj Low    float64
 3   Adj Close  float64
 4   Volume     int64  
dtypes: float64(4), int64(1)
memory usage: 399.5+ MB


In [8]:
print(f"Takes about 2.5 minutes to generate_features")

features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

Takes about 2.5 minutes to generate_features
⚡ Generating Decoupled Features (Benchmark: SPY)...


In [9]:
print(f"features_df.info():\n{features_df.info()}\n")
print(f"features_df.index.names:\n{features_df.index.names}\n")
print(f"macro_df.info():\n{macro_df.info()}\n")
print(f"macro_df.index.names:\n{macro_df.index.names}\n")

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 9503184 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-03-12 00:00:00'))
Data columns (total 13 columns):
 #   Column               Dtype  
---  ------               -----  
 0   ATR                  float64
 1   ATRP                 float64
 2   TRP                  float64
 3   RSI                  float64
 4   Mom_21               float64
 5   Consistency          float64
 6   IR_63                float64
 7   Beta_63              float64
 8   DD_21                float64
 9   Ret_1d               float64
 10  RollingStalePct      float64
 11  RollMedDollarVol     float64
 12  RollingSameVolCount  float64
dtypes: float64(13)
memory usage: 979.5+ MB
features_df.info():
None

features_df.index.names:
['Ticker', 'Date']

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 16156 entries, 1962-01-02 to 2026-03-12
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------      

In [ ]:
# Pre-flight Automated Audit Suite
checks = [
    SA.verify_math_integrity(),
    SA.verify_ranking_integrity(),
    SA.verify_vol_alignment_integrity(),
    SA.verify_feature_engineering_integrity(),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

print("=" * 85)
# Separate verify_marco_engine output from intertwine with other outputs

checks = [
    SA.verify_macro_engine(
        df_ohlcv=df_ohlcv,
        df_indices=df_indices,
        original_macro_df=macro_df,
        settings=GLOBAL_SETTINGS,
    ),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

In [11]:
print(
    "🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)"
)

# 1. Price Matrix
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)

# 2. Volatility Matrices (Unstack and Align)
# Using features_df (the first item from the tuple)
print("   - Unstacking ATRP...")
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)

print("   - Unstacking TRP...")
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# 3. Handle Data Gaps (Sanitize the Wide Matrices)
if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

# Forward fill up to the limit, then fill remaining with the "Disaster Detection" value
df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pre-computation Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)
print("   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.")

🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)
   - Unstacking ATRP...
   - Unstacking TRP...
✅ Pre-computation Complete. Tickers: 1580, Days: 16156
   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.


In [12]:
# 6. Instantiate engine (customize DataFrames as needed)
master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

In [13]:
# universe_subset=None means "Scan the whole market"
analyzer1, stage1_pack = create_walk_forward_analyzer(
    master_engine, universe_subset=None
)

print("🚀 Ready for Stage 1: Run Simulation for first filter.")
analyzer1.show()

🚀 Ready for Stage 1: Run Simulation for first filter.
DEBUG: 936 stocks passed filters on 2025-12-10


In [ ]:
# tick_price_252 = analyzer1.last_run.tickers
# tick_price_189 = analyzer1.last_run.tickers
# tick_price_126 = analyzer1.last_run.tickers
# tick_price_63 = analyzer1.last_run.tickers
# tick_price_42 = analyzer1.last_run.tickers
# tick_price_21 = analyzer1.last_run.tickers
# tick_price_10 = analyzer1.last_run.tickers

In [ ]:
# tick_sharpe_atrp_252 = analyzer1.last_run.tickers
# tick_sharpe_atrp_189 = analyzer1.last_run.tickers
# tick_sharpe_atrp_126 = analyzer1.last_run.tickers
# tick_sharpe_atrp_63 = analyzer1.last_run.tickers
# tick_sharpe_atrp_42 = analyzer1.last_run.tickers
# tick_sharpe_atrp_21 = analyzer1.last_run.tickers
# tick_sharpe_atrp_10 = analyzer1.last_run.tickers

In [58]:
union, intersection = set_operations(
    tick_sharpe_atrp_252,
    tick_sharpe_atrp_189,
    tick_sharpe_atrp_126,
    tick_sharpe_atrp_63,
    tick_sharpe_atrp_42,
    tick_sharpe_atrp_21,
    tick_sharpe_atrp_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_sharpe_atrp = intersection
# int_252_189_126 = intersection


Union (476 items): ['AA', 'ABNB', 'ACWI', 'ADI', 'ADSK', 'AEE', 'AEM', 'AEP', 'AER', 'AG', 'AGG', 'AGNC', 'AJG', 'ALB', 'AMAT', 'AMD', 'AME', 'AMGN', 'AMRZ', 'AMT', 'AMZN', 'AON', 'APA', 'APH', 'APLD', 'APP', 'AR', 'ARKK', 'ASML', 'ASTS', 'ASX', 'ATI', 'ATO', 'AU', 'AWK', 'AXON', 'AZN', 'B', 'BA', 'BALL', 'BBD', 'BBIO', 'BCS', 'BE', 'BG', 'BHP', 'BIL', 'BIV', 'BKLN', 'BKR', 'BLSH', 'BMY', 'BND', 'BNDX', 'BOXX', 'BR', 'BSV', 'BTI', 'BUD', 'BWXT', 'CACI', 'CAH', 'CASY', 'CAT', 'CB', 'CBOE', 'CCEP', 'CCJ', 'CDE', 'CEG', 'CF', 'CFG', 'CFLT', 'CGDV', 'CHD', 'CHRW', 'CIEN', 'CIFR', 'CL', 'CLS', 'CLX', 'CMCSA', 'CME', 'CMI', 'CMS', 'CNC', 'CNH', 'CNI', 'CNP', 'CNQ', 'COHR', 'COIN', 'COP', 'COR', 'COST', 'COWZ', 'CP', 'CRCL', 'CRL', 'CRS', 'CSL', 'CSX', 'CTVA', 'CVE', 'CVX', 'CW', 'DD', 'DE', 'DELL', 'DG', 'DGRO', 'DGX', 'DINO', 'DKNG', 'DLR', 'DOV', 'DOW', 'DPZ', 'DTE', 'DUK', 'DVA', 'EA', 'EBAY', 'ECL', 'ED', 'EEM', 'EFA', 'EFV', 'EFX', 'EIX', 'EMB', 'EME', 'EMXC', 'ENB', 'ENTG', 'EOG', 'EP

In [63]:
print(list_to_string(int_sharpe_atrp))

AEM, AG, ATI, BHP, BIL, BOXX, CASY, CIEN, FIX, FTI, GLW, JPST, KEYS, LITE, MINT, MTZ, NOC, NOK, PHYS, PULS, ROST, SGOV, SHV, SU, USFR


In [62]:
def list_to_string(items, separator=", ", last_separator=None):
    """
    Convert list to string with customizable separators

    Parameters:
    -----------
    items : list of strings
    separator : str, default ', '
        Separator between items
    last_separator : str, optional
        Special separator for last item (e.g., ' and ', ' or ')

    Returns:
    --------
    str : Formatted string

    Examples:
    ---------
    list_to_string(['a', 'b', 'c'])              # "a, b, c"
    list_to_string(['a', 'b', 'c'], ' | ')        # "a | b | c"
    list_to_string(['a', 'b', 'c'], ', ', ' and ')  # "a, b and c"
    """
    if not items:
        return ""

    if len(items) == 1:
        return str(items[0])

    if last_separator and len(items) > 1:
        return separator.join(items[:-1]) + last_separator + items[-1]

    return separator.join(str(item) for item in items)


# Usage examples
print(list_to_string(["a", "b", "c"]))  # a, b, c
print(list_to_string(["a", "b", "c"], " | "))  # a | b | c
print(list_to_string(["a", "b", "c"], ", ", " and "))  # a, b and c
print(list_to_string(["a", "b", "c"], ", ", ", and "))  # a, b, and c
print(list_to_string(["apple", "banana"], ", ", " and "))  # apple and banana

a, b, c
a | b | c
a, b and c
a, b, and c
apple and banana


In [59]:
union, intersection = set_operations(
    tick_price_252,
    tick_price_189,
    tick_price_126,
    tick_price_63,
    tick_price_42,
    tick_price_21,
    tick_price_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_price = intersection
# int_252_189_126 = intersection


Union (451 items): ['AA', 'ABNB', 'ADI', 'ADM', 'ADSK', 'AEE', 'AEM', 'AEP', 'AER', 'AES', 'AG', 'AJG', 'AKAM', 'ALAB', 'ALB', 'ALGN', 'AMAT', 'AMD', 'AME', 'AMGN', 'AMRZ', 'AMT', 'AMZN', 'ANET', 'AON', 'APA', 'APH', 'APLD', 'APP', 'AR', 'ARKK', 'ARM', 'ASML', 'ASTS', 'ASX', 'ATI', 'ATO', 'AU', 'AVAV', 'AVGO', 'AWK', 'AXON', 'AZN', 'B', 'BA', 'BALL', 'BBD', 'BBIO', 'BCS', 'BE', 'BG', 'BHP', 'BIDU', 'BIIB', 'BKR', 'BLD', 'BLSH', 'BMNR', 'BMY', 'BR', 'BTI', 'BUD', 'BURL', 'BWXT', 'C', 'CACI', 'CAH', 'CASY', 'CAT', 'CBOE', 'CCEP', 'CCI', 'CCJ', 'CDE', 'CEG', 'CELH', 'CF', 'CFG', 'CFLT', 'CHD', 'CHRW', 'CHTR', 'CHWY', 'CIEN', 'CIFR', 'CL', 'CLF', 'CLS', 'CMCSA', 'CME', 'CMI', 'CMS', 'CNC', 'CNH', 'CNI', 'CNP', 'CNQ', 'COHR', 'COIN', 'COP', 'CP', 'CPNG', 'CRCL', 'CRDO', 'CRL', 'CRS', 'CSL', 'CSX', 'CTRA', 'CVE', 'CVX', 'CW', 'DD', 'DE', 'DELL', 'DG', 'DGX', 'DINO', 'DKNG', 'DLR', 'DLTR', 'DOV', 'DOW', 'DPZ', 'DTE', 'DUK', 'DVA', 'DVN', 'EA', 'EBAY', 'EFX', 'EIX', 'EL', 'EME', 'EMN', 'EMXC'

In [61]:
union, intersection = set_operations(
    int_sharpe_atrp,
    int_price,
)
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_shp_atrp_price = intersection


Union (48 items): ['AEM', 'AG', 'APA', 'ATI', 'AU', 'BHP', 'BIL', 'BOXX', 'CASY', 'CDE', 'CIEN', 'CNQ', 'COHR', 'CRS', 'EQX', 'FCX', 'FIX', 'FN', 'FTI', 'GDXJ', 'GLW', 'HWM', 'IAG', 'JBHT', 'JPST', 'KEYS', 'LHX', 'LITE', 'MINT', 'MRNA', 'MTZ', 'NOC', 'NOK', 'PAAS', 'PHYS', 'PR', 'PULS', 'PWR', 'ROST', 'SCCO', 'SGOV', 'SHV', 'SIL', 'SU', 'USFR', 'VLO', 'WPM', 'XPO']
Intersection (16 items): ['AEM', 'AG', 'ATI', 'BHP', 'CASY', 'CIEN', 'FIX', 'FTI', 'GLW', 'KEYS', 'LITE', 'MTZ', 'NOC', 'NOK', 'PHYS', 'ROST']


In [ ]:
def set_operations(*lists):
    """
    Find sorted union and intersection of any number of lists of strings

    Parameters:
    -----------
    *lists : variable number of lists of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection)

    Examples:
    ---------
    union, intersection = set_operations(list1, list2)
    union, intersection = set_operations(list1, list2, list3, list4)
    """

    if not lists:
        return [], []

    # Convert each list to a set
    sets = [set(lst) for lst in lists]

    # Union: all unique elements from all lists
    union_set = set().union(*sets)  # or: sets[0].union(*sets[1:])
    union = sorted(union_set)

    # Intersection: common elements in ALL lists
    intersection_set = sets[0].intersection(*sets[1:]) if len(sets) > 1 else sets[0]
    intersection = sorted(intersection_set)

    return union, intersection


#

In [ ]:
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage1_pack = create_walk_forward_analyzer(
    master_engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
###############################
###############################

In [22]:
my_analyzer = analyzer1

my_res = SU.visualize_analyzer_structure(my_analyzer)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(259,))
[  2]   📈 benchmark_series (shape=(259,))
[  3]   🧮 normalized_plot_data (shape=(259, 200))
[  4]   📂 tickers (len=200)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]     📄 index_3 (str)
[  9]     📄 index_4 (str)
[ 10]     📄 index_5 (str)
[ 11]     📄 index_6 (str)
[ 12]     📄 index_7 (str)
[ 13]     📄 index_8 (str)
[ 14]     📄 index_9 (str)
[ 15]     📄 index_10 (str)
[ 16]     📄 index_11 (str)
[ 17]     📄 index_12 (str)
[ 18]     📄 index_13 (str)
[ 19]     📄 index_14 (str)
[ 20]     📄 index_15 (str)
[ 21]     📄 index_16 (str)
[ 22]     📄 index_17 (str)
[ 23]     📄 index_18 (str)
[ 24]     📄 index_19 (str)
[ 25]     📄 index_20 (str)
[ 26]     📄 index_21 (str)
[ 27]     📄 index_22 (str)
[ 28]     📄 index_23 (str)
[ 29]     📄 index_24 (str)
[ 30]     📄 index_25 (str)
[ 31]     📄 index_26 (str)
[ 32]     📄 index_27 (str)
[ 33]     📄 index_28 (str)
[

In [27]:
def set_operations(list1, list2):
    """
    Find sorted union and intersection of two lists of strings

    Parameters:
    -----------
    list1, list2 : list of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection)
    """

    # Convert to sets for operations
    set1 = set(list1)
    set2 = set(list2)

    # Union: all unique elements from both lists
    union = sorted(set1 | set2)  # or set1.union(set2)

    # Intersection: common elements in both lists
    intersection = sorted(set1 & set2)  # or set1.intersection(set2)

    return union, intersection


# Example usage
list_a = ["apple", "banana", "cherry", "date", "elderberry"]
list_b = ["banana", "date", "fig", "grape", "apple"]

union, intersection = set_operations(list_a, list_b)

print(f"List 1: {list_a}")
print(f"List 2: {list_b}")
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

List 1: ['apple', 'banana', 'cherry', 'date', 'elderberry']
List 2: ['banana', 'date', 'fig', 'grape', 'apple']

Union (7 items): ['apple', 'banana', 'cherry', 'date', 'elderberry', 'fig', 'grape']
Intersection (3 items): ['apple', 'banana', 'date']


In [ ]:
union, intersection = set_operations(list_a, list_b)

In [ ]:
tick_sharpe_trp_252 = SU.peek(4, my_res)

In [ ]:
analyzer1.last_run.tickers
analyzer1.last_run.start_date
analyzer1.last_run.holding_end_date

In [ ]:
# 3. Post-flight Reconciliation
audit = SA.verify_analyzer_short(my_analyzer)
if not audit.ok:
    print(f"🚨 ALERT: {audit.msg}")
    # You could trigger a notification or log the failure here

In [ ]:
audit = SA.verify_analyzer_long(my_analyzer, n_tickers=5)

In [ ]:
# Takes 4 seconds to run, checks selected tickers from analyzer1
SA.audit_feature_engineering_integrity(my_analyzer, mode="last_run")

### Audit Every Tickers in features_df, Takes 3 minutes 

In [ ]:
# # Takes 3 minutes to run, checks every tickers ~1580 tickers
# SA.audit_feature_engineering_integrity(my_analyzer, mode="system")

### Export Ticker's OHLCV and Features

In [ ]:
f_name_excel = OUTPUT_DIR / "Audit_Verification_Report.xlsx"

SU.export_audit_to_excel(audit_pack=my_analyzer.last_run, filename=f_name_excel)

In [ ]:
f_name_csv = OUTPUT_DIR / "all_tickers_data_stacked.csv"

# Single call replaces your 3 cells
file_path = SU.export_last_run_tickers_data_to_csv(
    analyzer=my_analyzer,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename=f_name_csv,
)

In [ ]:
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=my_analyzer.last_run.tickers,
    date_start=my_analyzer.last_run.start_date,
    date_end=analyzer1.last_run.holding_end_date,
    verbose=False,
)

In [ ]:
_tic = "NVDA"
_start_date = "2025-03-12"
_end_date = "2026-03-12"
print(features_df.loc[_tic][_start_date:_end_date])
# features_df.loc[_tic][_start_date:_end_date][["Ret_1d", "Consistency"]]

In [ ]:
result = SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=[_tic],
    date_start=_start_date,
    date_end=_end_date,
    verbose=False,
)
print(result[_tic])